# Предобучение BookLM: шаг, оптимизатор, 16 бит и возобновление

**Вопрос.** Прогон упал на середине. Можно ли поднять его из файла так, чтобы следующий шаг совпал с шагом
прогона, который не падал, до последнего бита? И что должно лежать в этом файле?

Ноутбук проходит на CPU путь главы 7 на уменьшенной BookLM: те же блоки, RoPE, RMSNorm, групповое внимание
и связанные веса, только уже и мельче. Корпус крошечный — полтора десятка абзацев о самой книге,
записанных прямо в ноутбуке. На обычном ноутбуке всё выполняется меньше чем за минуту.

**Ожидаем.**

1. Полная BookLM собирается с $6\,622\,464$ параметрами, а после правила инициализации главы стартовая ошибка
   отличается от $\ln 8192\approx9{,}01$ меньше чем на $0{,}1$.
2. Градиент маленькой сети совпадает с центральной конечной разностью в float64 с относительной ошибкой меньше
   $10^{-7}$, а во float32 та же проверка ошибается на проценты.
3. AdamW, написанный руками, повторяет `torch.optim.AdamW` и по параметрам, и по моментам $m$, $v$.
4. Градиенты в FP16 при большом масштабе ошибки переполняются; шаг с переполнением не трогает ни параметры,
   ни $m$, $v$, ни счётчики, а масштаб уменьшается вдвое.
5. «Обучить $k$ шагов, сохранить, загрузить в новые объекты, обучить ещё $j$» даёт побитно те же параметры,
   что непрерывный прогон.
6. Рядом с весами лежит манифест `booklm-base-step-N`, и загрузчик с чужим родителем падает.
7. Если при возобновлении забыть состояние оптимизатора, первая ошибка после загрузки всё равно совпадёт, а
   сравнение параметров поломку поймает.

По дороге ноутбук проходит все десять проверок из конца главы; в коде они помечены комментариями
`# проверка N`.

Профиль `BOOKLM_PROFILE=full` берёт настоящую конфигурацию BookLM и две тысячи историй её корпуса. На CPU её шаг
занимает около секунды, так что весь бюджет из $4\,043$ шагов — опыт для ускорителя; в автоматическую
проверку этот профиль не входит.

In [ ]:
import hashlib
import io
import json
import math
import os
import platform
import random
import subprocess
import tempfile
import time
import unicodedata
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(min(4, os.cpu_count() or 1))   # одно и то же число потоков у всех прогонов
print("python:", platform.python_version(), "| numpy:", np.__version__, "| torch:", torch.__version__,
      "| потоков:", torch.get_num_threads())

PROFILE = os.environ.get("BOOKLM_PROFILE", "cpu")     # cpu — уменьшенная BookLM; full — настоящая
STEPS = int(os.environ.get("BOOKLM_STEPS", "240"))
RUN_DIR = Path(os.environ.get("BOOKLM_RUN_DIR") or tempfile.mkdtemp(prefix="booklm-07-"))
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("профиль:", PROFILE, "| шагов:", STEPS, "| каталог запуска:", RUN_DIR)

## 1. BookLM из главы 4 и правило инициализации

Код блока — тот же, что в главе 4. Добавлены две вещи, которые нужны упаковке документов из раздела о данных:
номер позиции считается внутри своего документа, а маска внимания разрешает смотреть только назад и только в
свой документ. RMSNorm считает не ниже чем во float32: во float64 она так и остаётся во float64, иначе
проверка градиента в разделе 4 потеряла бы точность.

In [ ]:
def rope(x, positions, base=10000.0):
    # x: [B, H, T, d_h]; positions: [B, T]; пара (x_2k, x_2k+1) поворачивается на угол p * theta_k
    d_h = x.size(-1)
    theta = base ** (-torch.arange(0, d_h, 2, dtype=torch.float32) / d_h)      # [d_h/2]
    angles = positions.to(torch.float32)[:, None, :, None] * theta               # [B, 1, T, d_h/2]
    cos, sin = angles.cos().to(x.dtype), angles.sin().to(x.dtype)
    x0, x1 = x[..., 0::2], x[..., 1::2]
    return torch.stack((x0 * cos - x1 * sin, x0 * sin + x1 * cos), dim=-1).flatten(-2)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.weight, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        wide = x.to(torch.promote_types(x.dtype, torch.float32))                  # не ниже float32
        scale = torch.rsqrt(wide.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (wide * scale).to(x.dtype) * self.weight


class GroupedQueryAttention(nn.Module):
    def __init__(self, d, hq, hkv):
        super().__init__()
        self.hq, self.hkv, self.dh = hq, hkv, d // hq
        self.q = nn.Linear(d, hq * self.dh, bias=False)
        self.k = nn.Linear(d, hkv * self.dh, bias=False)
        self.v = nn.Linear(d, hkv * self.dh, bias=False)
        self.out = nn.Linear(d, d, bias=False)                                     # W_O

    def forward(self, x, positions, allowed):                                      # x: [B, T, d]
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.hq, self.dh).transpose(1, 2)
        k = self.k(x).view(b, t, self.hkv, self.dh).transpose(1, 2)
        v = self.v(x).view(b, t, self.hkv, self.dh).transpose(1, 2)
        q, k = rope(q, positions), rope(k, positions)
        r = self.hq // self.hkv
        k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)       # GQA
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dh)                      # [B, H, T, T]
        weights = scores.masked_fill(~allowed[:, None], float("-inf")).softmax(dim=-1)
        return self.out((weights @ v).transpose(1, 2).reshape(b, t, -1))


class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.up = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)                                 # W_down

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class BookLMBlock(nn.Module):
    def __init__(self, d, hq, hkv, d_ff):
        super().__init__()
        self.norm_1, self.attention = RMSNorm(d), GroupedQueryAttention(d, hq, hkv)
        self.norm_2, self.ffn = RMSNorm(d), SwiGLU(d, d_ff)

    def forward(self, x, positions, allowed):
        x = x + self.attention(self.norm_1(x), positions, allowed)
        return x + self.ffn(self.norm_2(x))


def doc_positions(doc):
    # doc: [B, T] — номер документа у каждой позиции; результат — номер позиции внутри документа
    idx = torch.arange(doc.size(1)).expand_as(doc)
    first = torch.ones_like(doc, dtype=torch.bool)
    first[:, 1:] = doc[:, 1:] != doc[:, :-1]
    start = torch.where(first, idx, torch.zeros_like(idx)).cummax(dim=1).values
    return idx - start


class BookLM(nn.Module):
    def __init__(self, vocab_size=8192, d=256, layers=6, hq=8, hkv=2, d_ff=768):
        super().__init__()
        self.tokens = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList(BookLMBlock(d, hq, hkv, d_ff) for _ in range(layers))
        self.norm = RMSNorm(d)
        self.output = nn.Linear(d, vocab_size, bias=False)
        self.output.weight = self.tokens.weight                                    # связанные веса

    def forward(self, ids, doc=None, drop=0.0):                                    # [B, T] -> [B, T, V]
        b, t = ids.shape
        doc = torch.zeros_like(ids) if doc is None else doc
        causal = torch.tril(torch.ones(t, t, dtype=torch.bool))
        allowed = causal & (doc[:, :, None] == doc[:, None, :])                    # [B, T, T]
        positions = doc_positions(doc)
        h = F.dropout(self.tokens(ids), p=drop, training=drop > 0)
        for block in self.blocks:
            h = block(h, positions, allowed)
        return self.output(self.norm(h))


@torch.no_grad()
def init_booklm_(model, std=0.02):
    # booklm-base-init: матрицы N(0, 0.02^2), W_O и W_down N(0, (0.02/sqrt(2L))^2), усиления RMSNorm = 1
    branch_std = std / math.sqrt(2 * max(len(model.blocks), 1))
    for name, p in model.named_parameters():                                       # общий тензор — один раз
        if p.dim() == 1:
            nn.init.ones_(p)
        elif name.endswith(("out.weight", "down.weight")):
            nn.init.normal_(p, std=branch_std)
        else:
            nn.init.normal_(p, std=std)


@torch.no_grad()
def random_text_loss(model, V, batch=4, length=128, seed=0):
    ids = torch.randint(0, V, (batch, length + 1), generator=torch.Generator().manual_seed(seed))
    logits = model(ids[:, :-1])
    return F.cross_entropy(logits.flatten(0, 1), ids[:, 1:].flatten()).item()


BOOKLM = dict(vocab_size=8192, d=256, layers=6, hq=8, hkv=2, d_ff=768)
torch.manual_seed(0)
full = BookLM(**BOOKLM)
assert sum(p.numel() for p in full.parameters()) == 6_622_464                  # проверка 1: паспорт
assert full.tokens.weight is full.output.weight                                # таблица и выход — один тензор
loss_default = random_text_loss(full, 8192)
torch.manual_seed(0)
init_booklm_(full)
loss_init = random_text_loss(full, 8192)
print(f"BookLM: стартовая ошибка {loss_default:.1f} с инициализацией PyTorch, "
      f"{loss_init:.3f} по правилу главы; ln 8192 = {math.log(8192):.3f}")
assert loss_default > 100                                                      # N(0, 1) в связанной таблице
assert abs(loss_init - math.log(8192)) < 0.1                                   # проверка 2
del full

С инициализацией PyTorch по умолчанию таблица токенов заполнена числами из $\mathcal N(0;1)$. Веса связаны,
поэтому та же таблица стоит на выходе, логиты получаются огромными, а ошибка уходит в сотни. Правило главы
возвращает её к рулетке из $8192$ ячеек.

## 2. Крошечный корпус: дубли, отложенные документы и упаковка в окна

Токенизатор здесь байтовый: номер токена — байт UTF-8, номер $256$ — конец документа. Настоящий
`booklm-tokenizer-v1` для опытов этой главы не нужен, а байты не требуют файла словаря. Перед упаковкой
нормализуем тексты и выбрасываем точные копии по SHA-256; четыре документа откладываем для валидации.

In [ ]:
DOCS = [
    "Мама мыла раму, а папа читал газету. Мама мыла раму каждое утро, и рама блестела на солнце.",
    "Внимание вычисляет, на какие прошлые токены смотреть. Каждый запрос сравнивается с ключами, "
    "и доли внимания в сумме дают единицу.",
    "Шаг обучения состоит из шести действий: взять пакет, сделать прямой проход, сравнить логиты с целями, "
    "получить одно число ошибки, пройти назад и сдвинуть параметры против градиента.",
    "Случайная модель ошибается примерно на логарифм размера словаря. Если стартовая ошибка намного больше, "
    "ищите поломку в инициализации.",
    "Градиент имеет ту же форму, что и параметры. Обратный проход считает его примерно за цену двух прямых.",
    "Оптимизатор AdamW хранит для каждого параметра два момента: среднее градиента и среднее его квадрата. "
    "Без них продолжить прогон нельзя.",
    "Расписание скорости начинается с разогрева, потом скорость плавно спадает по косинусу до малого "
    "значения в конце прогона.",
    "В формате FP16 большой градиент превращается в бесконечность, а маленький в ноль. Масштаб ошибки "
    "поднимает маленькие градиенты, а при переполнении шаг пропускают.",
    "Контрольная точка хранит параметры, моменты, номер шага, масштаб ошибки, состояние генераторов "
    "и позицию загрузчика.",
    "Валидационную ошибку считают на отложенных текстах. Разница меньше двух стандартных ошибок ещё не "
    "улучшение.",
    "Упаковка складывает документы в окна одинаковой длины. Маска документов не даёт токену смотреть в "
    "чужой документ.",
    "Перплексия показывает, из скольких равновероятных вариантов модель в среднем выбирает следующий токен.",
    "Мама  мыла раму, а папа читал газету.  Мама мыла раму каждое утро, и рама блестела на солнце.",
    "Бит на байт делит полную ошибку на длину текста в байтах. Так сравнивают модели с разными словарями.",
    "Связанные веса делают таблицу токенов и выходную матрицу одним тензором. Его градиент складывается "
    "из двух вкладов.",
    "Ограничение нормы градиента делают после снятия масштаба. Норму до ограничения пишут в журнал "
    "каждого шага.",
    "Мама мыла раму, а внимание вычисляет доли. Шаг обучения сдвигает параметры против градиента.",
    "Оптимизатор хранит моменты, а сохранение хранит позицию загрузчика и номер шага.",
    "Ошибка случайной модели близка к логарифму словаря, а перплексия равна размеру словаря.",
    "Масштаб ошибки уменьшают вдвое, когда градиент в FP16 переполнился, и пропускают этот шаг.",
]
N_VALID = 4                                                    # последние четыре документа — отложенные


def normalize(text):
    return " ".join(unicodedata.normalize("NFC", text).split())


def dedup(docs):
    seen, kept = set(), []
    for doc in map(normalize, docs):
        key = hashlib.sha256(doc.encode("utf-8")).hexdigest()
        if key not in seen:
            seen.add(key)
            kept.append(doc)
    return kept


def tale_paragraphs(n=2000):
    # профиль full: первые n историй корпуса BookLM, если ноутбук запущен из репозитория
    for root in (Path("models/booklm/data"), Path("../models/booklm/data")):
        path = root / "sft.jsonl"
        if path.exists():
            with path.open(encoding="utf-8") as f:
                return [json.loads(line)["story"] for _, line in zip(range(n), f)]
    return []


clean = dedup(DOCS)
assert len(clean) == len(DOCS) - 1                             # копия с лишними пробелами ушла
train_docs, valid_docs = clean[:-N_VALID], clean[-N_VALID:]
if PROFILE == "full":                                           # истории BookLM: 5 % откладываем
    tales = [d for d in dedup(tale_paragraphs()) if d not in clean]
    n_held = max(len(tales) // 20, 1) if tales else 0
    train_docs = tales[:len(tales) - n_held] + train_docs
    valid_docs = tales[len(tales) - n_held:] + valid_docs
assert not set(train_docs) & set(valid_docs)                   # отложенных текстов в обучении нет

EOS, IGNORE = 256, -100


def encode(text):
    return list(text.encode("utf-8"))


def decode(ids, errors="replace"):
    return bytes(i for i in ids if i < 256).decode("utf-8", errors=errors)


def pack(docs, window):
    # документы подряд, после каждого EOS; окна по window = T + 1 номеров; хвост отбрасываем
    stream, owner = [], []
    for n, doc in enumerate(docs, start=1):
        ids = encode(doc) + [EOS]
        stream += ids
        owner += [n] * len(ids)
    cut = len(stream) // window * window
    return (torch.tensor(stream[:cut]).view(-1, window),
            torch.tensor(owner[:cut]).view(-1, window))


def make_batch(rows, owners):
    # X, Y со сдвигом; цель после EOS не учим — это первый токен чужого документа
    x, y, doc = rows[:, :-1], rows[:, 1:].clone(), owners[:, :-1]
    y[x == EOS] = IGNORE
    return x, y, doc


if PROFILE == "full":
    CFG = dict(BOOKLM)
    T, BATCH, WARMUP, LR_MAX, LR_MIN = 512, 8, min(200, max(STEPS // 10, 1)), 3e-4, 3e-5
else:                                                          # уменьшенная BookLM
    CFG = dict(vocab_size=257, d=64, layers=2, hq=4, hkv=2, d_ff=192)
    T, BATCH, WARMUP, LR_MAX, LR_MIN = 64, 8, 20, 3e-3, 3e-4
TRAIN_ROWS, TRAIN_OWNERS = pack(train_docs, T + 1)
VALID_ROWS, VALID_OWNERS = pack(valid_docs, T + 1)
VALID = make_batch(VALID_ROWS, VALID_OWNERS)
print(f"обучение: {len(train_docs)} документов, {TRAIN_ROWS.shape[0]} окон по {T + 1}; "
      f"валидация: {len(valid_docs)} документа, {VALID_ROWS.shape[0]} окон")

Проверим маску документов тестом влияния из главы. Перепишем первый документ окна и посмотрим на выходы
второго документа того же окна. С маской они не должны сдвинуться ни на бит; без неё второй документ читает
первый.

In [ ]:
torch.manual_seed(0)
probe = BookLM(**CFG)
init_booklm_(probe)
row = next(i for i in range(len(TRAIN_ROWS)) if TRAIN_OWNERS[i, :-1].unique().numel() >= 2)
x0, _, doc0 = make_batch(TRAIN_ROWS[row:row + 1], TRAIN_OWNERS[row:row + 1])
first_doc = doc0[0, 0]
changed = x0.clone()
changed[doc0 == first_doc] = (changed[doc0 == first_doc] + 7) % 256    # переписали первый документ
later = doc0[0] != first_doc
with torch.no_grad():
    shift_masked = (probe(x0, doc0) - probe(changed, doc0))[0, later].abs().max().item()
    shift_causal = (probe(x0) - probe(changed))[0, later].abs().max().item()
print(f"сдвиг выходов второго документа: с маской {shift_masked}, только причинная маска {shift_causal:.3f}")
assert shift_masked == 0.0                                     # проверка 4
assert shift_causal > 1e-3

## 3. Один шаг обучения руками

Шесть действий шага из раздела 1 главы: пакет, прямой проход, сравнение с целями, одно число ошибки,
обратный проход, сдвиг параметров. С SGD шаг можно проверить прямо формулой $\theta-\eta g$.

In [ ]:
def train_step(model, batch, opt):
    x, y, doc = batch                                          # 1. пакет [B, T]
    opt.zero_grad(set_to_none=True)
    logits = model(x, doc)                                     # 2. прямой проход [B, T, V]
    loss = F.cross_entropy(logits.flatten(0, 1), y.flatten(),  # 3. сравнение с Y
                           ignore_index=IGNORE)                # 4. одно число
    loss.backward()                                            # 5. обратный проход
    opt.step()                                                 # 6. theta - eta * g
    return loss.item()


def fresh_model(seed=0):
    torch.manual_seed(seed)
    model = BookLM(**CFG)
    init_booklm_(model)
    return model


model = fresh_model()
V = CFG["vocab_size"]
start = random_text_loss(model, V)
print(f"{'BookLM' if PROFILE == 'full' else 'уменьшенная BookLM'}: {sum(p.numel() for p in model.parameters()):,} параметров, "
      f"стартовая ошибка {start:.3f}, ln V = {math.log(V):.3f}")
assert abs(start - math.log(V)) < 0.1

batch = make_batch(TRAIN_ROWS[:BATCH], TRAIN_OWNERS[:BATCH])
ETA = 0.02 if PROFILE == "full" else 0.1
opt = torch.optim.SGD(model.parameters(), lr=ETA)
losses = [train_step(model, batch, opt) for _ in range(5)]
print("ошибка на одном пакете:", " -> ".join(f"{l:.3f}" for l in losses))
assert all(b < a for a, b in zip(losses, losses[1:]))
assert all(p.grad.shape == p.shape for p in model.parameters())   # градиент той же формы

w = model.output.weight
before = w.detach().clone()
train_step(model, batch, opt)
step_gap = (w.detach() - (before - ETA * w.grad)).abs().max().item()
print(f"шаг SGD против формулы theta - eta*g: {step_gap:.1e}")
assert step_gap < 1e-7                                         # шаг = формула (PyTorch умножает и вычитает
                                                               # одной операцией, отсюда последний бит)

Пакет BookLM может не поместиться в память, и тогда его режут на микропакеты и копят градиент. Делить сумму
ошибок нужно на общее число учебных токенов пакета, а не брать среднее средних: у микропакетов разное число
целей, потому что цели после EOS и заполнитель не считаются.

In [ ]:
def accumulate(model, x, y, doc, denom):
    model.zero_grad(set_to_none=True)
    for j in range(len(x)):                                    # микропакет — одно окно
        logits = model(x[j:j + 1], doc[j:j + 1]).flatten(0, 1)
        loss = F.cross_entropy(logits, y[j], ignore_index=IGNORE, reduction="sum")
        (loss / denom(j)).backward()
    return [p.grad.clone() for p in model.parameters()]


model = fresh_model()
x, y, doc = make_batch(TRAIN_ROWS[:2], TRAIN_OWNERS[:2])
y = y.clone()
y[1, 5:] = IGNORE                                              # во втором окне всего несколько целей
counts = (y != IGNORE).sum(dim=1)
model.zero_grad(set_to_none=True)
F.cross_entropy(model(x, doc).flatten(0, 1), y.flatten(), ignore_index=IGNORE).backward()
whole = [p.grad.clone() for p in model.parameters()]
right = accumulate(model, x, y, doc, lambda j: counts.sum())
wrong = accumulate(model, x, y, doc, lambda j: counts[j] * len(x))   # среднее средних
gap_right = max((a - b).abs().max().item() for a, b in zip(whole, right))
gap_wrong = max((a - b).abs().max().item() for a, b in zip(whole, wrong))
print(f"целей в микропакетах: {counts.tolist()}; расхождение с целым пакетом: "
      f"верно {gap_right:.1e}, среднее средних {gap_wrong:.1e}")
assert gap_right < 1e-6                                        # проверка 5
assert gap_wrong > 1e-4

## 4. Проверка градиента конечными разностями

Сначала сеть из двух матриц из раздела об обратном проходе: градиент руками по цепному правилу и повороты
«ручек» на $\pm\varepsilon$. Потом то же для нескольких координат уменьшенной BookLM, где градиент считает
autograd PyTorch. Всё во float64: во float32 разность двух близких ошибок теряет большую часть значащих цифр.

In [ ]:
xs = np.array([1.0, 2.0])                                      # [2]
W = np.array([[1.0, -1.0], [0.5, 0.5]])                        # [2, 2]
U = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 1.0]])            # [3, 2]
target = 0


def forward(W, U, xs):
    h = W @ xs
    r = np.maximum(h, 0)                                       # ReLU
    a = U @ r                                                  # [3] логиты
    return h, r, a, np.log(np.exp(a).sum()) - a[target]


h, r, a, L = forward(W, U, xs)
g_a = np.exp(a) / np.exp(a).sum() - np.eye(3)[target]         # p - one_hot
dU = np.outer(g_a, r)
dW = np.outer((U.T @ g_a) * (h > 0), xs)
assert abs(L - 2.2989) < 1e-4
assert np.allclose(dW, [[0, 0], [0.8996, 1.7993]], atol=1e-4)


def numeric_grad(P, loss, eps):
    G = np.zeros_like(P)
    for i in np.ndindex(P.shape):
        old = P[i]
        P[i] = old + eps; up = loss()
        P[i] = old - eps; down = loss()
        P[i] = old
        G[i] = (up - down) / (2 * eps)
    return G


def finite_difference_error(dtype):
    W_, U_, x_ = W.astype(dtype), U.astype(dtype), xs.astype(dtype)
    loss = lambda: forward(W_, U_, x_)[3]
    nW, nU = numeric_grad(W_, loss, dtype(1e-6)), numeric_grad(U_, loss, dtype(1e-6))
    return max(np.abs(nW - dW).max() / np.abs(dW).max(), np.abs(nU - dU).max() / np.abs(dU).max())


err64, err32 = finite_difference_error(np.float64), finite_difference_error(np.float32)
print(f"сеть из двух матриц: относительная ошибка {err64:.1e} во float64, {err32:.3f} во float32")
assert err64 < 1e-7                                            # проверка 3
assert err32 > 1e-4                                            # float32 для этой проверки не годится

In [ ]:
model64 = fresh_model().double()
x, y, doc = make_batch(TRAIN_ROWS[:2, :17], TRAIN_OWNERS[:2, :17])   # два коротких окна


def loss64():
    return F.cross_entropy(model64(x, doc).flatten(0, 1), y.flatten(), ignore_index=IGNORE)


loss64().backward()
named = dict(model64.named_parameters())
probes = ["tokens.weight", "blocks.0.attention.q.weight", "blocks.0.attention.out.weight",
          "blocks.1.ffn.down.weight", "blocks.1.norm_2.weight", "norm.weight"]
worst = 0.0
with torch.no_grad():
    for name in probes:
        p = named[name]
        i = np.unravel_index(p.grad.abs().argmax().item(), p.shape)     # самая заметная координата
        old, eps = p[i].item(), 1e-6
        p[i] = old + eps; up = loss64().item()
        p[i] = old - eps; down = loss64().item()
        p[i] = old
        numeric = (up - down) / (2 * eps)
        rel = abs(numeric - p.grad[i].item()) / abs(p.grad[i].item())
        worst = max(worst, rel)
        print(f"{name:32s} {str(tuple(int(k) for k in i)):12s} autograd {p.grad[i].item():+.6e}  "
              f"разность {numeric:+.6e}")
print(f"наибольшая относительная ошибка: {worst:.1e}")
assert worst < 1e-6
del model64

## 5. AdamW руками против `torch.optim.AdamW`

Функция `adamw_step` — та же, что в разделе об оптимизаторах: моменты, поправка смещения, нормировка и
затухание мимо нормировки. Возьмём две одинаковые копии уменьшенной BookLM. Одну двигает PyTorch, другую — наша
функция, с теми же группами параметров: матрицы с затуханием, усиления RMSNorm без него. Связанная таблица
попадает в группы один раз.

In [ ]:
def adamw_step(theta, g, m, v, s, lr, beta1=0.9, beta2=0.95, eps=1e-8, wd=0.0):
    # theta, g, m, v: одной формы; s — номер шага, начиная с 1
    m = beta1 * m + (1 - beta1) * g
    v = beta2 * v + (1 - beta2) * g * g
    m_hat = m / (1 - beta1 ** s)                               # поправка смещения
    v_hat = v / (1 - beta2 ** s)
    update = m_hat / (v_hat.sqrt() + eps) + wd * theta         # затухание мимо нормировки
    return theta - lr * update, m, v


def param_groups(model, weight_decay):
    decay, no_decay = [], []
    for _, p in model.named_parameters():                      # общий тензор — один раз
        (decay if p.ndim >= 2 else no_decay).append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def loss_on(model, batch):
    x, y, doc = batch
    return F.cross_entropy(model(x, doc).flatten(0, 1), y.flatten(), ignore_index=IGNORE)


theirs, mine = fresh_model(), fresh_model()
opt = torch.optim.AdamW(param_groups(theirs, 0.1), lr=3e-3, betas=(0.9, 0.95), eps=1e-8)
wd_of = {id(p): g["weight_decay"] for g in param_groups(mine, 0.1) for p in g["params"]}
state = {id(p): (torch.zeros_like(p), torch.zeros_like(p)) for p in mine.parameters()}
for s in range(1, 6):
    batch = make_batch(TRAIN_ROWS[s * BATCH:(s + 1) * BATCH], TRAIN_OWNERS[s * BATCH:(s + 1) * BATCH])
    opt.zero_grad()
    loss_on(theirs, batch).backward()
    opt.step()
    mine.zero_grad()
    loss_on(mine, batch).backward()
    with torch.no_grad():
        for p in mine.parameters():
            m, v = state[id(p)]
            new, m, v = adamw_step(p, p.grad, m, v, s, lr=3e-3, wd=wd_of[id(p)])
            p.copy_(new)
            state[id(p)] = (m, v)
gap_theta = max((p - q).abs().max().item() for p, q in zip(mine.parameters(), theirs.parameters()))
gap_m = max((state[id(p)][0] - opt.state[q]["exp_avg"]).abs().max().item()
            for p, q in zip(mine.parameters(), theirs.parameters()))
gap_v = max((state[id(p)][1] - opt.state[q]["exp_avg_sq"]).abs().max().item()
            for p, q in zip(mine.parameters(), theirs.parameters()))
steps = {int(opt.state[q]["step"]) for q in theirs.parameters()}
print(f"после 5 шагов: |theta| {gap_theta:.1e}, |m| {gap_m:.1e}, |v| {gap_v:.1e}, step в PyTorch {steps}")
assert gap_theta < 1e-5 and gap_m < 1e-6 and gap_v < 1e-8 and steps == {5}

Расписание «разогрев и косинус» из того же раздела. `LambdaLR` нумерует шаги с нуля, а формула главы — с
единицы; без сдвига на единицу первый шаг прогона прошёл бы со скоростью $0$.

In [ ]:
def lr_at(s, warmup, total, lr_max, lr_min):
    # s — номер шага оптимизатора, начиная с 1
    if s <= warmup:
        return lr_max * s / warmup
    progress = (s - warmup) / (total - warmup)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * progress))


SCHED = dict(warmup=WARMUP, total=STEPS, lr_max=LR_MAX, lr_min=LR_MIN)
lr_lambda = lambda i: lr_at(i + 1, **SCHED) / LR_MAX          # LambdaLR: шаг s получает lr_lambda(s - 1)
assert math.isclose(lr_at(WARMUP, **SCHED), LR_MAX) and math.isclose(lr_at(STEPS, **SCHED), LR_MIN)
bk = dict(warmup=200, total=4043, lr_max=3e-4, lr_min=3e-5)    # настоящий прогон BookLM
assert abs(lr_at(2121, **bk) - 1.65e-4) < 1e-7 and math.isclose(lr_at(1, **bk), 1.5e-6)

## 6. Шестнадцать бит: переполнение и масштаб ошибки

У FP16 пять бит порядка: наибольшее число $65\,504$, наименьшее нормальное $2^{-14}$. Ускоритель, который
считает обратный проход в FP16, получает градиенты в этом формате. Здесь мы считаем на CPU во float32, а
формат градиента воспроизводим честно: после обратного прохода каждый градиент переводим в FP16 и обратно.
Значения, которые FP16 не вмещает, становятся нулём или бесконечностью ровно так же, как на ускорителе.

Сначала посмотрим, что делает с градиентами уменьшенной BookLM масштаб ошибки $S$: ошибку умножают на $S$
перед обратным проходом, градиенты потом делят на $S$.

In [ ]:
f16 = torch.finfo(torch.float16)
assert (f16.max, f16.tiny) == (65504.0, 2.0 ** -14)
to16 = lambda value: torch.tensor(value).to(torch.float16).item()
assert to16(70000.0) == float("inf") and to16(1e-8) == 0.0
w16 = torch.tensor(0.02, dtype=torch.bfloat16)                 # BF16: малое обновление тонет
assert (w16 - torch.tensor(3e-5, dtype=torch.bfloat16)).item() == w16.item()

model = fresh_model()
batch = make_batch(TRAIN_ROWS[:BATCH], TRAIN_OWNERS[:BATCH])
model.zero_grad()
loss_on(model, batch).backward()
grads = torch.cat([p.grad.flatten() for p in model.parameters()])
grads = grads[grads != 0]                                       # строки таблицы вне пакета не считаем
print(f"градиенты float32: от {grads.abs().min():.1e} до {grads.abs().max():.1e}")
print("   S       в ноль    в inf")
fractions = {}
for k in (0, 8, 16, 24, 32):
    S = 2.0 ** k
    in16 = (grads * S).to(torch.float16)
    fractions[k] = ((in16 == 0).float().mean().item(), torch.isinf(in16).float().mean().item())
    print(f"2^{k:<3d}  {fractions[k][0]:8.4%}  {fractions[k][1]:8.4%}")
assert fractions[0][0] > fractions[16][0] and fractions[16] == (0.0, 0.0)
assert fractions[32][1] > 0                                     # слишком большой масштаб переполняет

Без масштаба пропадают самые малые градиенты, меньше $2^{-25}$: в нашем опыте это сотые доли процента, но
именно эти координаты и так двигаются еле-еле. При $S=2^{16}$ ничего не пропало и ничего не переполнилось, а при
$2^{32}$ часть значений ушла в бесконечность. Подбирать $S$ руками не нужно: динамический масштабировщик
начинает с большого значения, при переполнении пропускает шаг и делит $S$ пополам, а после $n_{\text{рост}}$
удачных шагов подряд удваивает. Запустим его со слишком большого $S=2^{30}$ и сравним с
`torch.amp.GradScaler` на тех же переполнениях.

In [ ]:
class DynamicScaler:                                            # те же поля, что у torch.amp.GradScaler
    def __init__(self, init_scale=2.0 ** 16, growth_interval=2000):
        self.S, self.interval, self.good = init_scale, growth_interval, 0

    def unscale_and_check(self, params):                        # True: нашли inf или NaN
        found_inf = False
        for p in params:
            p.grad.mul_(1.0 / self.S)                           # снимаем масштаб во float32
            found_inf |= not torch.isfinite(p.grad).all().item()
        return found_inf

    def update(self, found_inf):
        self.good = 0 if found_inf else self.good + 1
        if found_inf:
            self.S /= 2
        elif self.good == self.interval:
            self.S, self.good = self.S * 2, 0

    def state_dict(self):
        return {"scale": self.S, "growth_factor": 2.0, "backoff_factor": 0.5,
                "growth_interval": self.interval, "_growth_tracker": self.good}


def grads_through_fp16(model):
    for p in model.parameters():                                # градиент хранится в FP16
        p.grad = p.grad.to(torch.float16).to(torch.float32)


def make_amp_run(seed=0):
    model = fresh_model(seed)
    opt = torch.optim.AdamW(param_groups(model, 0.1), lr=3e-3, betas=(0.9, 0.95))
    return model, opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)


def snapshot(model, opt, sched):                                # всё, что живёт дольше шага
    tensors = [p.detach().clone() for p in model.parameters()]
    tensors += [t.clone() for st in opt.state.values() for t in st.values()]
    return tensors, sched.last_epoch


def my_step(model, opt, sched, scaler, batch):
    opt.zero_grad(set_to_none=True)
    (loss_on(model, batch) * scaler.S).backward()
    grads_through_fp16(model)
    found_inf = scaler.unscale_and_check(model.parameters())
    if not found_inf:
        opt.step()                                              # шаг и оба счётчика — только здесь
        sched.step()
    scaler.update(found_inf)
    return found_inf


def torch_step(model, opt, sched, scaler, batch):
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss_on(model, batch)).backward()
    grads_through_fp16(model)
    before = scaler.get_scale()
    scaler.step(opt)                                            # при inf шага нет
    scaler.update()
    skipped = scaler.get_scale() < before
    if not skipped:
        sched.step()
    return skipped


mine, theirs = make_amp_run(), make_amp_run()
my_scaler = DynamicScaler(init_scale=2.0 ** 30, growth_interval=5)
torch_scaler = torch.amp.GradScaler("cpu", init_scale=2.0 ** 30, growth_interval=5)
amp_batch = lambda i: make_batch(TRAIN_ROWS[i % 5 * BATCH:(i % 5 + 1) * BATCH],
                                 TRAIN_OWNERS[i % 5 * BATCH:(i % 5 + 1) * BATCH])
log = []
for i in range(30):
    log.append((my_step(*mine, my_scaler, amp_batch(i)), torch_step(*theirs, torch_scaler, amp_batch(i)),
                my_scaler.S))
print("пропуски:", "".join("x" if s else "." for s, _, _ in log))
print("масштаб: ", ", ".join(f"2^{int(math.log2(S))}" for _, _, S in log))
assert all(a == b for a, b, _ in log)                           # пропуски на тех же шагах
assert all(torch.equal(p, q) for p, q in zip(mine[0].parameters(), theirs[0].parameters()))
assert my_scaler.state_dict() == torch_scaler.state_dict()
assert mine[2].last_epoch == 30 - sum(s for s, _, _ in log)    # планировщик считает только шаги

i = 30
while True:                                                     # ждём переполнения после удачных шагов
    before, S_before = snapshot(*mine), my_scaler.S
    if my_step(*mine, my_scaler, amp_batch(i)):
        break
    i += 1
after = snapshot(*mine)
assert before[1] == after[1]                                    # проверка 6: планировщик стоит,
assert all(torch.equal(a, b) for a, b in zip(before[0], after[0]))   # параметры, m, v и step те же,
assert my_scaler.S == S_before / 2                              # а масштаб вдвое меньше
print(f"шаг {i + 1}: переполнение при S = 2^{int(math.log2(S_before))}; параметры, m, v, step и "
      f"планировщик не изменились, S = 2^{int(math.log2(my_scaler.S))}")

Первые шаги масштабировщик только уменьшает $S$: каждое переполнение — пропущенный шаг. Потом прогон идёт, и
каждые пять удачных шагов $S$ удваивается, пока снова не упрётся в переполнение. Наш класс и `GradScaler`
пропустили одни и те же шаги и пришли к тем же параметрам бит в бит.

## 7. Прогон, контрольная точка и побитное возобновление

Соберём полный прогон уменьшенной BookLM: AdamW с группами, разогрев и косинус, масштабировщик, ограничение
нормы, прореживание (dropout), которое тянет числа из генератора, и загрузчик, который перемешивает окна по
эпохам. В файл сохранения кладём всё из таблицы главы о его содержимом: параметры, состояние оптимизатора,
планировщик, масштабировщик, позицию загрузчика и состояния трёх генераторов. Пишем атомарно: сначала во
временный файл, потом одно переименование. Упади процесс посреди записи — старое сохранение останется целым.

In [ ]:
class Loader:                                                    # позиция: (seed, epoch, index)
    def __init__(self, rows, owners, batch_size, seed):
        self.rows, self.owners, self.b = rows, owners, batch_size
        self.seed, self.epoch, self.index = seed, 0, 0

    def next_batch(self):
        if self.index + self.b > len(self.rows):
            self.epoch, self.index = self.epoch + 1, 0
        g = torch.Generator().manual_seed(self.seed + self.epoch)
        pick = torch.randperm(len(self.rows), generator=g)[self.index:self.index + self.b]
        self.index += self.b
        return make_batch(self.rows[pick], self.owners[pick])

    def state_dict(self):
        return dict(seed=self.seed, epoch=self.epoch, index=self.index)

    def load_state_dict(self, state):
        self.seed, self.epoch, self.index = state["seed"], state["epoch"], state["index"]


DROP, CLIP, GROWTH = 0.1, 1.0, 50
PARTS = ("model", "optimizer", "scheduler", "scaler", "loader")


def make_run(seed):
    model = fresh_model(seed)
    opt = torch.optim.AdamW(param_groups(model, 0.1), lr=LR_MAX, betas=(0.9, 0.95), eps=1e-8)
    return {"model": model, "optimizer": opt,
            "scheduler": torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda),
            "scaler": torch.amp.GradScaler("cpu", growth_interval=GROWTH),
            "loader": Loader(TRAIN_ROWS, TRAIN_OWNERS, BATCH, seed=seed)}


def run_step(run):
    model, opt, scaler = run["model"], run["optimizer"], run["scaler"]
    x, y, doc = run["loader"].next_batch()
    lr = opt.param_groups[0]["lr"]
    logits = model(x, doc, drop=DROP)
    loss = F.cross_entropy(logits.flatten(0, 1), y.flatten(), ignore_index=IGNORE)
    scaler.scale(loss).backward()                               # ошибка * S
    scaler.unscale_(opt)                                        # / S и поиск inf
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP).item()   # норма до ограничения
    scaler.step(opt)
    scaler.update()
    run["scheduler"].step()
    opt.zero_grad(set_to_none=True)
    return {"loss": loss.item(), "grad_norm": norm, "lr": lr, "tokens": int((y != IGNORE).sum())}


@torch.no_grad()
def evaluate(model, batch):
    x, y, doc = batch
    nll = F.cross_entropy(model(x, doc).flatten(0, 1), y.flatten(), ignore_index=IGNORE,
                          reduction="none").view(y.shape)
    mask = (y != IGNORE).float()
    per_window = (nll * mask).sum(1) / mask.sum(1)
    sigma = per_window.std() / len(per_window) ** 0.5           # стандартная ошибка среднего
    return ((nll * mask).sum() / mask.sum()).item(), sigma.item()


def write_atomic(path, data):
    fd, tmp = tempfile.mkstemp(dir=os.path.dirname(path), suffix=".tmp")
    with os.fdopen(fd, "wb") as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())                                    # байты на диске
    os.replace(tmp, path)                                       # одно переименование


def save_state(run, path, step):
    state = {name: run[name].state_dict() for name in PARTS}
    np_state = list(np.random.get_state())
    np_state[1] = np_state[1].tolist()
    state["rng"] = {"torch": torch.get_rng_state(), "numpy": np_state, "python": random.getstate()}
    state["step"] = step
    buffer = io.BytesIO()
    torch.save(state, buffer)
    write_atomic(path, buffer.getvalue())


def load_state(run, path, skip=()):
    state = torch.load(path, weights_only=True)                 # без исполнения чужого кода
    for name in PARTS:
        if name not in skip:
            run[name].load_state_dict(state[name])
    if "rng" not in skip:
        rng = state["rng"]
        torch.set_rng_state(rng["torch"])
        np_state = list(rng["numpy"])
        np_state[1] = np.array(np_state[1], dtype=np.uint32)
        np.random.set_state(tuple(np_state))
        random.setstate(rng["python"])
    return state["step"]

Непрерывный прогон: `STEPS` шагов, каждые `EVAL_EVERY` шагов — валидационная ошибка на отложенных документах
вместе с номером шага и числом увиденных токенов. Валидация не тянет чисел из генераторов, поэтому прогон с
ней и без неё один и тот же.

In [ ]:
EVAL_EVERY = max(STEPS // 8, 1)
K = STEPS // 2                                                  # здесь «упадёт» второй прогон
parent_weights = {n: p.detach().clone() for n, p in make_run(0)["model"].named_parameters()}

torch.manual_seed(SEED)
t0 = time.perf_counter()
whole = make_run(0)
journal, tokens_seen = [], 0
val0 = evaluate(whole["model"], VALID)
evals = [(0, 0, *val0)]
for step in range(1, STEPS + 1):
    row = run_step(whole)
    tokens_seen += BATCH * T
    journal.append({"step": step, "tokens_seen": tokens_seen, **row})
    if step % EVAL_EVERY == 0 or step == STEPS:
        evals.append((step, tokens_seen, *evaluate(whole["model"], VALID)))
seconds = time.perf_counter() - t0
reference = [p.detach().clone() for p in whole["model"].parameters()]
print(f"{STEPS} шагов за {seconds:.1f} с; ошибка на пакете {journal[0]['loss']:.3f} -> {journal[-1]['loss']:.3f}")
print(" шаг  токенов   вал.ошибка  sigma")
for s, tok, val, sig in evals:
    print(f"{s:4d} {tok:8d}   {val:.4f}    {sig:.4f}")

window = max(STEPS // 10, 1)
smooth = lambda i: sum(r["loss"] for r in journal[i - window:i]) / window
assert smooth(STEPS) < smooth(window)                          # обучающая ошибка падает
assert evals[-1][2] < evals[0][2] - 2 * evals[-1][3]           # проверка 8: валидация ниже стартовой,
for i in range(1, len(evals)):
    best = min(v for _, _, v, _ in evals[:i])
    assert evals[i][2] < best + 2 * evals[i][3]                 # и не выросла над лучшей больше чем на 2 sigma

Теперь второй прогон «падает» после шага $K$. Сохраняем состояние, удаляем объекты, создаём новые с другой
затравкой (так ведёт себя новый процесс, которому никто не передал старые генераторы) и грузим файл.
Сравниваем параметры с непрерывным прогоном через `torch.equal`: никаких допусков.

In [ ]:
CKPT = RUN_DIR / f"train-state-step-{K}.pt"


def resumed(skip=()):
    torch.manual_seed(SEED)
    first = make_run(0)
    for _ in range(K):
        run_step(first)
    save_state(first, CKPT, K)
    del first                                                   # процесс упал
    torch.manual_seed(123)
    second = make_run(123)                                      # новые объекты, другие числа
    step = load_state(second, CKPT, skip)
    tail = [run_step(second) for _ in range(STEPS - step)]
    return second, tail


again, tail = resumed()
state = torch.load(CKPT, weights_only=True)
assert set(state) == {*PARTS, "rng", "step"}
assert int(state["optimizer"]["state"][0]["step"]) == K and state["scheduler"]["last_epoch"] == K
assert [p.name for p in RUN_DIR.iterdir() if p.suffix == ".tmp"] == []   # временного файла не осталось
same = all(torch.equal(p, r) for p, r in zip(again["model"].parameters(), reference))
assert same and [r["loss"] for r in tail] == [r["loss"] for r in journal[K:]]   # проверка 7
assert again["scaler"].get_scale() == whole["scaler"].get_scale()
print(f"сохранение на шаге {K}: {CKPT.stat().st_size:,} байт; после загрузки ещё {STEPS - K} шагов, "
      f"параметры совпали побитно: {same}")

## 8. Измерить: перплексия, бит на байт и контрольные продолжения

Перплексия $e^{\overline L}$ читается как число равновероятных вариантов. Токен здесь — байт, поэтому бит на
байт почти равен ошибке в битах; EOS байтом не считаем. Пять фиксированных начал продолжаем жадно на 30
токенов. Возобновлённый прогон — это повторный запуск того же рецепта, и продолжения у него должны выйти те же,
как и хеш логитов контрольного пакета.

In [ ]:
val_loss, val_sigma = evals[-1][2], evals[-1][3]
ppl = math.exp(val_loss)
n_targets = int((VALID[1] != IGNORE).sum())
n_bytes = int(((VALID[1] != IGNORE) & (VALID[1] != EOS)).sum())
bits_per_byte = lambda loss: loss * n_targets / (n_bytes * math.log(2))
bpb, bpb_start = bits_per_byte(val_loss), bits_per_byte(evals[0][2])
print(f"валидация: {val_loss:.4f} ± {val_sigma:.4f} ната, перплексия {ppl:.1f} из {V}, "
      f"бит на байт {bpb:.3f} (у родителя {bpb_start:.3f})")
assert ppl < math.exp(evals[0][2]) and bpb < bpb_start

PROMPTS = ["Мама мыла", "Внимание вычисляет", "Шаг обучения", "Оптимизатор", "Масштаб ошибки"]


@torch.no_grad()
def greedy(model, prompt, n=30):
    ids = encode(prompt)
    for _ in range(n):
        logits = model(torch.tensor([ids[-T:]]))
        ids.append(int(logits[0, -1].argmax()))
    return ids[len(encode(prompt)):]


@torch.no_grad()
def batch_hash(model):
    x, _, doc = VALID
    return hashlib.sha256(model(x, doc).numpy().tobytes()).hexdigest()


samples = {p: greedy(whole["model"], p) for p in PROMPTS}
for p, ids in samples.items():
    print(f"{p!r:22s} -> {decode(ids, errors='ignore')!r}")
assert samples == {p: greedy(again["model"], p) for p in PROMPTS}   # проверка 9
CONTROL_HASH = batch_hash(whole["model"])
assert CONTROL_HASH == batch_hash(again["model"])

Продолжения пока похожи на лепет: сто тысяч параметров и три килобайта текста дают частые слоги, но не слова.
Для проверки 9 это не помеха. Она спрашивает другое: выдаёт ли тот же рецепт тот же текст. Если завтра
продолжения изменятся, а код и данные нет, где-то потерялась затравка или состояние генератора.

## 9. Выпускаемые веса и манифест `booklm-base-step-N`

Для дообучения в главе 8 и для запуска в главе 14 моменты и позиция загрузчика не нужны. Выпускаем отдельный
файл: только параметры в BF16, связанный тензор один раз. Хеш считаем по именам в алфавитном порядке. Манифест
по образцу `BOOKLM.md` указывает родителя и его SHA-256, рецепт прогона и измерения. В профиле CPU модель
уменьшенная, поэтому к имени добавлен суффикс `-cpu-mini`: такие веса нельзя спутать с настоящей BookLM.

In [ ]:
def release(model):                                             # BF16; связанный тензор — один раз
    return {name: p.detach().to(torch.bfloat16) for name, p in model.named_parameters()}


def weights_sha256(weights):
    h = hashlib.sha256()
    for name in sorted(weights):
        h.update(name.encode() + weights[name].view(torch.int16).numpy().tobytes())
    return h.hexdigest()


def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True,
                                       stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "unknown"


suffix = "" if PROFILE == "full" else "-cpu-mini"
ARTIFACT_ID = f"booklm-base-step-{STEPS}{suffix}"
weights = release(whole["model"])
weights_path = RUN_DIR / f"{ARTIFACT_ID}.pt"
torch.save(weights, weights_path)
n_params = sum(t.numel() for t in weights.values())
assert sum(t.numel() * t.element_size() for t in weights.values()) == 2 * n_params
PARENT_ID = f"booklm-base-init{suffix}"
manifest = {
    "id": ARTIFACT_ID,
    "parents": {PARENT_ID: weights_sha256({n: t.to(torch.bfloat16) for n, t in parent_weights.items()})},
    "created_by": "notebooks/07_pretraining.ipynb",
    "format": "torch-state-dict-bf16",
    "format_version": 1,
    "sha256": hashlib.sha256(weights_path.read_bytes()).hexdigest(),
    "weights_sha256": weights_sha256(weights),
    "tokenizer_sha256": hashlib.sha256(b"bytes-utf8; 256 = eos").hexdigest(),
    "code_commit": git_commit(),
    "environment": f"python {platform.python_version()}, torch {torch.__version__}, CPU, "
                   f"{torch.get_num_threads()} потока",
    "config": {**CFG, "window": T, "parameters": n_params},
    "recipe": {"seed": SEED, "steps": STEPS, "batch": BATCH, "optimizer": "AdamW", "betas": [0.9, 0.95],
               "eps": 1e-8, "weight_decay": 0.1, "schedule": {"warmup": WARMUP, "lr_max": LR_MAX,
               "lr_min": LR_MIN, "shape": "cosine"}, "clip_norm": CLIP, "dropout": DROP,
               "precision": "fp32, GradScaler cpu"},
    "step": STEPS,
    "tokens_seen": tokens_seen,
    "metrics": {"train_loss_last": round(journal[-1]["loss"], 4),
                "validation": [{"step": s, "tokens_seen": tok, "loss": round(v, 4), "sigma": round(sg, 4)}
                               for s, tok, v, sg in evals],
                "perplexity": round(ppl, 2), "bits_per_byte": round(bpb, 4)},
    "samples": {p: decode(ids, errors="ignore") for p, ids in samples.items()},
    "control_batch_sha256": CONTROL_HASH,
    "training_state": {"file": CKPT.name, "step": K,
                       "sha256": hashlib.sha256(CKPT.read_bytes()).hexdigest()},
}
manifest_path = RUN_DIR / f"{ARTIFACT_ID}.manifest.json"
write_atomic(manifest_path, json.dumps(manifest, ensure_ascii=False, indent=2).encode("utf-8"))


def load_release(manifest_path, parent, parent_id=PARENT_ID):
    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
    expected = manifest["parents"][parent_id]
    if weights_sha256(release(parent)) != expected:
        raise ValueError(f"родитель не тот: ждали {expected[:12]}...")
    weights = torch.load(RUN_DIR / f"{manifest['id']}.pt", weights_only=True)
    if weights_sha256(weights) != manifest["weights_sha256"]:
        raise ValueError("файл весов изменился")
    return weights


loaded = load_release(manifest_path, make_run(0)["model"])
assert weights_sha256(loaded) == manifest["weights_sha256"]
try:                                                             # проверка 10
    load_release(manifest_path, make_run(1)["model"])
    raise AssertionError("загрузчик принял чужого родителя")
except ValueError as err:
    print("чужой родитель отклонён:", err)
print(f"{ARTIFACT_ID}: {weights_path.stat().st_size:,} байт, {n_params:,} параметров")
print(json.dumps({"id": manifest["id"], "parents": list(manifest["parents"]),
                  "step": manifest["step"], "tokens_seen": manifest["tokens_seen"]},
                 ensure_ascii=False))

## 10. Сломанный вариант: возобновили без состояния оптимизатора

Частая ошибка — сохранить веса и номер шага, а моменты AdamW забыть. Проверим такое возобновление двумя
способами. Наивная проверка сравнивает первую ошибку после загрузки. Честная — параметры после нескольких
шагов, как в разделе 7.

In [ ]:
def verify_resume(run, tail):
    # честная проверка: параметры и журнал после загрузки совпадают с непрерывным прогоном
    if not all(torch.equal(p, r) for p, r in zip(run["model"].parameters(), reference)):
        gap = max((p - r).abs().max().item() for p, r in zip(run["model"].parameters(), reference))
        raise AssertionError(f"возобновление разошлось с непрерывным прогоном: {gap:.3g}")


broken, broken_tail = resumed(skip=("optimizer",))
naive_ok = broken_tail[0]["loss"] == journal[K]["loss"]
print("наивная проверка (первая ошибка после загрузки совпала):", naive_ok)
try:
    verify_resume(broken, broken_tail)
    raise RuntimeError("сломанное возобновление прошло проверку")
except AssertionError as err:
    print("честная проверка:", err)
assert naive_ok                                                 # первая ошибка считается до шага
verify_resume(again, tail)                                      # верное возобновление проходит


def first_move(skip):
    # первый шаг после загрузки того же файла: длина сдвига параметров и скорость eta этого шага
    torch.manual_seed(123)
    second = make_run(123)
    load_state(second, CKPT, skip)
    start = [p.detach().clone() for p in second["model"].parameters()]
    lr = run_step(second)["lr"]
    move = math.sqrt(sum(((p - s) ** 2).sum().item() for p, s in zip(second["model"].parameters(), start)))
    return move, lr


(move_right, lr_right), (move_broken, lr_broken) = first_move(()), first_move(("optimizer",))
print(f"первый шаг после загрузки с моментами: eta = {lr_right:.2e}, сдвиг {move_right:.4f}, "
      f"в единицах eta {move_right / lr_right:.0f}")
print(f"первый шаг после загрузки без них:     eta = {lr_broken:.2e}, сдвиг {move_broken:.4f}, "
      f"в единицах eta {move_broken / lr_broken:.0f}")
assert math.isclose(lr_right, lr_at(K + 1, **SCHED), rel_tol=1e-6)
assert math.isclose(lr_broken, lr_at(1, **SCHED), rel_tol=1e-6)   # скорость тоже забыта: снова разогрев
assert move_broken / lr_broken > 1.5 * move_right / lr_right

Первая ошибка после загрузки совпала: её считают до шага, по тем же весам и тому же пакету. Расходится сам
шаг, и сразу по двум причинам. Текущая скорость $\eta$ тоже лежит в состоянии оптимизатора, поэтому новый AdamW
сделал первый шаг со скоростью из начала разогрева, которую ему записал новый планировщик. А моменты начались
с $m=v=0$, и после поправки смещения шаг вышел близким к $\eta\cdot\operatorname{sign} g$ по каждой
координате: в единицах $\eta$ он в несколько раз длиннее шага прогона с накопленными моментами. Ошибка после
такого скачка быстро возвращается к прежней кривой, и на графике его легко принять за шум. Поэтому
возобновление проверяют по параметрам.

## 11. Код главы 7 из `labs/ch07`

Длинные листинги главы лежат в отдельных файлах `labs/ch07/*.py`. В книге остались короткие версии, а здесь
каждый файл выполняется целиком: при загрузке он проходит все свои `assert`. Порядок файлов — порядок разделов
главы. После каждого файла сверяем его с тем, что ноутбук уже посчитал выше, или печатаем главные числа.

Файлы загружаются через `importlib` по явному пути. У каждого своё пространство имён, поэтому его `BookLM`,
`adamw_step` и `lr_at` не перекрывают функции ноутбука. Путь ищется от текущего каталога вверх, так что
ноутбук работает и из `notebooks/`, и из корня репозитория. Раздел стоит в конце, потому что файлы сами
переставляют общий генератор `torch.manual_seed`: прогоны выше от этого не зависят.

In [ ]:
import importlib.util


def find_labs_ch07():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "labs" / "ch07"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError("labs/ch07 не найден: запустите ноутбук внутри репозитория книги")


LABS_CH07 = find_labs_ch07()


def load_lab(name):
    lab_path = LABS_CH07 / f"{name}.py"
    spec = importlib.util.spec_from_file_location(f"ch07_{name}", lab_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)   # здесь выполняются все проверки файла
    return module


LAB_FILES = sorted(p.stem for p in LABS_CH07.glob("*.py"))
print("labs:", LABS_CH07, "| файлов:", len(LAB_FILES))

### 11.1. Один шаг обучения на TinyLM и забытый `zero_grad`

Раздел «Один шаг обучения: шесть наблюдаемых действий». Файл `labs/ch07/train_step_tinylm.py` — полная
версия листинга «Один шаг обучения как функция». Он делает пять шагов на одном пакете и проверяет, что ошибка
каждый раз падает, а градиент каждого параметра имеет ту же форму, что сам параметр. Потом сверяет шаг
`torch.optim.SGD` с формулой $\theta - \eta g$ и ставит опыт с забытым `zero_grad`: норма градиента второго
шага у TinyLM равна $2{,}00$ вместо $1{,}00$.

In [ ]:
step_lab = load_lab("train_step_tinylm")

assert all(b < a for a, b in zip(step_lab.losses, step_lab.losses[1:]))   # ошибка падает каждый шаг
assert 1.8 < step_lab.norms[1] / step_lab.norms[0] < 2.3                  # без zero_grad градиенты сложились
print(f"TinyLM, ошибка за пять шагов: {step_lab.losses[0]:.3f} -> {step_lab.losses[-1]:.3f}")
print(f"норма градиента второго шага: с zero_grad {step_lab.norms[0]:.2f}, без него {step_lab.norms[1]:.2f}")

### 11.2. Шаг по логитам, правило инициализации и забытый `zero_grad`

Раздел «Один шаг обучения: шесть наблюдаемых действий». Файл `labs/ch07/booklm_init_and_step.py` делает шаг
по логитам руками и сверяет его с autograd, собирает полную BookLM главы 4 и проверяет стартовую ошибку до и
после правила инициализации. В конце он ставит опыт с забытым `zero_grad`: градиент второго шага
складывается с первым, и его норма вырастает примерно вдвое.

In [ ]:
init_lab = load_lab("booklm_init_and_step")

assert sum(p.numel() for p in init_lab.booklm.parameters()) == 6_622_464   # проверка 1
assert init_lab.booklm.tokens.weight is init_lab.booklm.output.weight
print(f"норма градиента второго шага: с zero_grad {init_lab.right:.2f}, без него {init_lab.wrong:.2f}")

### 11.3. Почти-дубли, упаковка и маска документов

Раздел «Данные: от сырого текста к окнам с границами». Файл `labs/ch07/minhash_toy_corpus.py` проводит десять
документов через весь конвейер: точные дубли, MinHash с полосами LSH, фильтры, отложенную валидацию и проверку
загрязнения. Файл `labs/ch07/packing_document_mask.py` упаковывает документы в окна и ставит тот же тест
влияния, что раздел 2 ноутбука, только на одном слое внимания.

In [ ]:
minhash_lab = load_lab("minhash_toy_corpus")
for stage, docs in minhash_lab.funnel:
    print(f"{stage:>13}: {len(docs)} документов")

packing_lab = load_lab("packing_document_mask")
assert packing_lab.shift(packing_lab.mask) == 0.0 == shift_masked              # проверка 4: здесь и выше
print(f"сдвиг второго документа без маски документов: {packing_lab.shift(packing_lab.causal_only):.4f}")

### 11.4. Обратный проход руками и autograd

Раздел «Обратное распространение: градиент руками и запись графа». Файл `labs/ch07/backprop_autograd.py`
берёт ту же сеть из двух матриц, что раздел 4 ноутбука, и проверяет градиент центральной разностью во
float64 и float32. Потом он строит класс `Value`, который записывает граф, сравнивает сигнал через
двенадцать слоёв с остаточной связью и без неё, складывает два вклада в градиент связанной таблицы и считает
операции прямого и обратного прохода.

In [ ]:
backprop_lab = load_lab("backprop_autograd")

assert abs(backprop_lab.L - 2.2989) < 1e-4                       # та же сеть, что в разделе 4
assert backprop_lab.err64 < 1e-7 < 1e-3 < backprop_lab.err32     # проверка 3
print(f"относительная ошибка разности: float64 {backprop_lab.err64:.1e}, float32 {backprop_lab.err32:.1e}")

### 11.5. SGD, момент, AdamW и расписание скорости

Раздел «Оптимизатор: от SGD к AdamW и расписание скорости». Три файла:

- `labs/ch07/optim_sgd_momentum_adamw.py` — траектории на овраге, сверка момента и `adamw_step` с
  `torch.optim`, группы параметров BookLM;
- `labs/ch07/optim_lr_schedule.py` — разогрев и косинус, сверка с `LambdaLR` и поломка «без сдвига на единицу»;
- `labs/ch07/optim_three_on_tinylm.py` — SGD, момент и AdamW на TinyLM, когда ошибку умножили на 1, 100 и 0,01.

Функции `adamw_step` и `lr_at` есть и в файлах, и в разделе 5 ноутбука. Ниже проверяем, что это одни и те же
формулы.

In [ ]:
optim_lab = load_lab("optim_sgd_momentum_adamw")
assert optim_lab.sizes == [6_619_136, 3_328]          # с затуханием и без него
gen = torch.Generator().manual_seed(7)
theta, g = torch.randn(5, generator=gen), torch.randn(5, generator=gen)
m, v = torch.zeros(5), torch.zeros(5)
for s in range(1, 4):
    ours = adamw_step(theta, g, m, v, s, lr=1e-2, wd=0.1)
    theirs = optim_lab.adamw_step(theta, g, m, v, s, lr=1e-2, wd=0.1)
    assert all(torch.equal(a, b) for a, b in zip(ours, theirs))
    theta, m, v = ours

schedule_lab = load_lab("optim_lr_schedule")
assert all(math.isclose(schedule_lab.lr_at(s, **schedule_lab.bk), lr_at(s, **schedule_lab.bk))
           for s in range(1, 4044))
assert schedule_lab.wrong[0] == 0.0                   # без сдвига первый шаг идёт с lr = 0

three_lab = load_lab("optim_three_on_tinylm")
print("ошибка последних 20 шагов при масштабе 1 / 100 / 0,01:")
for name, row in three_lab.table.items():
    print(f"  {name:>8}: " + " / ".join(f"{x:.3f}" for x in row))

### 11.6. Шестнадцать бит, масштабировщик и порядок ограничения нормы

Раздел «Шаг в 16 битах: точность, масштаб и ограничение нормы». Файл
`labs/ch07/precision_scaler_full_step.py` разбирает форматы FP32, FP16 и BF16 по битам и ставит опыт с
бесконечностью в градиенте на TinyLM. Свой `DynamicScaler` он сверяет с `torch.amp.GradScaler` в полном шаге
смешанной точности, а затем ограничивает норму до снятия масштаба. AdamW такую ошибку почти не замечает, SGD
получает шаг в сотни раз короче.

In [ ]:
precision_lab = load_lab("precision_scaler_full_step")

assert precision_lab.scaler.S == 2.0 ** 16          # проверка 6: после пропуска S упал вдвое и вырос обратно
print(f"сдвиг весов за прогон: AdamW {precision_lab.right:.3f} против {precision_lab.wrong:.3f}, "
      f"SGD {precision_lab.sgd_right:.3f} против {precision_lab.sgd_wrong:.4f}")

### 11.7. Контрольная точка, выпуск весов и бюджет

Раздел «Длинный прогон: сохранить, измерить, рассчитать бюджет». Файл
`labs/ch07/run_checkpoint_release_eval.py` повторяет разделы 7–9 ноутбука на TinyLM и скелете BookLM. В нём
атомарная запись, побитное возобновление и четыре поломки, выпуск весов в BF16 с манифестом, перплексия, бит на
байт и бюджет Chinchilla. В конце файл меряет скорость шага BookLM на вашем процессоре. Это число только
печатается, проверять его нечем.

In [ ]:
run_lab = load_lab("run_checkpoint_release_eval")

assert run_lab.gap == 0.0                             # проверка 7: возобновление совпало до бита
for name, (gap, _, _) in run_lab.gaps.items():
    print(f"поломка «{name}»: расхождение параметров {gap:.3f}")
print(f"бюджет Chinchilla: {run_lab.D:,} токенов, {run_lab.steps} шагов; "
      f"на этом CPU около {run_lab.D / run_lab.speed / 3600:.1f} ч")

# Файлы labs/ch07 вне ноутбука: проверки листингов главы и два долгих опыта (часы на CPU).
NOT_IN_NOTEBOOK = {"step_data_backprop_checks", "optim_precision_run_checks",  # assert к листингам
                   "decoding", "mup_width_transfer"}                          # опыты о выборке токена и о μP
IN_NOTEBOOK = {"backprop_autograd", "booklm_init_and_step", "minhash_toy_corpus", "optim_lr_schedule",
               "optim_sgd_momentum_adamw", "optim_three_on_tinylm", "packing_document_mask",
               "precision_scaler_full_step", "run_checkpoint_release_eval", "train_step_tinylm"}
assert set(LAB_FILES) == IN_NOTEBOOK | NOT_IN_NOTEBOOK, set(LAB_FILES) ^ (IN_NOTEBOOK | NOT_IN_NOTEBOOK)  # новый файл — новый подраздел выше

## Связь с промышленной системой

| В ноутбуке | В настоящем прогоне |
|---|---|
| `make_run` и `PARTS` | состояние тренера: модель, оптимизатор, планировщик, `GradScaler`, загрузчик |
| `save_state` / `load_state` | `torch.distributed.checkpoint` с `get_state_dict` / `set_state_dict` для FSDP и DDP |
| `write_atomic` | запись во временный каталог и переименование; у `torch.distributed.checkpoint` файл метаданных пишется последним |
| `Loader.state_dict` | `StatefulDataLoader` из `torchdata` или номер последнего образца в Megatron-LM |
| `DynamicScaler` | `torch.amp.GradScaler`: те же поля `scale`, `_growth_tracker`, `growth_interval` |
| `lr_at` через `LambdaLR` | `get_cosine_with_min_lr_schedule_with_warmup` в `transformers` |
| `adamw_step` | `torch.optim.AdamW` (`exp_avg`, `exp_avg_sq`, `step`) |
| манифест `booklm-base-step-N` | карточка модели и метаданные `safetensors` рядом с хешем родителя |

## Самостоятельное задание

1. Возобновите прогон без планировщика: `resumed(skip=("scheduler",))`. Предскажите до запуска, на каком
   шаге после загрузки скорость `lr` в журнале впервые разойдётся с непрерывным прогоном, и проверьте.
   Почему ответ изменится, если `LambdaLR` создавать уже после загрузки оптимизатора?
2. Запустите ноутбук с `BOOKLM_STEPS=1000`. Какая проверка упадёт и почему? Посмотрите на таблицу валидации:
   с какого шага модель начинает заучивать три килобайта обучающего текста? Что из рецепта вы поменяете,
   чтобы проверка снова проходила при том же числе шагов?